# 04 Export for Frontend

This notebook converts the heavier modeling and monitoring CSV outputs into lightweight static JSON contracts for the React frontend. The files are written to `frontend/public/data/`, where the app can fetch them directly without a live API server.

## 0. Imports

The export step uses pandas for aggregation, Python's standard JSON tools for serialization, and SciPy only to recompute the paired t-test p-value used in the simulation summary.

In [1]:
import json
import math
from datetime import datetime, timedelta
from pathlib import Path
from textwrap import dedent
from typing import Any, Dict, Iterable, List

import numpy as np
import pandas as pd
from scipy import stats

PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
FRONTEND_DATA_DIR = PROJECT_ROOT / "frontend" / "public" / "data"

JSON_FILES = [
    "zones.json",
    "summary.json",
    "predictions.json",
    "drift_metrics.json",
    "retrain_events.json",
    "weekly_comparison.json",
    "zone_performance.json",
    "time_period_analysis.json",
]

## 1. Load All CSVs

The export notebook reads every processed artifact from the modeling and monitoring notebooks, plus the TLC zone lookup table for service-zone metadata. Row counts and date ranges are printed as a quick integrity check before writing JSON.

In [2]:
predictions_v1 = pd.read_csv(PROCESSED_DIR / "predictions_v1.csv", parse_dates=["timestamp"])
simulation_results = pd.read_csv(
    PROCESSED_DIR / "simulation_results.csv",
    parse_dates=["week_start", "week_end"],
)
drift_metrics = pd.read_csv(
    PROCESSED_DIR / "drift_metrics.csv",
    parse_dates=["week_start", "week_end"],
)
retrain_events = pd.read_csv(PROCESSED_DIR / "retrain_events.csv", parse_dates=["timestamp"])
weekly_predictions = pd.read_csv(
    PROCESSED_DIR / "weekly_predictions.csv",
    parse_dates=["timestamp"],
)
zone_lookup = pd.read_csv(RAW_DIR / "taxi_zone_lookup.csv").rename(
    columns={"LocationID": "zone_id", "Zone": "zone_name", "Borough": "borough"}
)
zone_lookup["zone_id"] = zone_lookup["zone_id"].astype(int)

loaded_frames = {
    "predictions_v1": predictions_v1,
    "simulation_results": simulation_results,
    "drift_metrics": drift_metrics,
    "retrain_events": retrain_events,
    "weekly_predictions": weekly_predictions,
}

for name, frame in loaded_frames.items():
    print(f"{name}: {len(frame):,} rows")
    date_columns = [column for column in frame.columns if "timestamp" in column or column.startswith("week_")]
    for column in date_columns:
        print(f"  {column}: {frame[column].min()} to {frame[column].max()}")

predictions_v1: 6,040 rows
  timestamp: 2024-03-19 10:00:00 to 2024-03-31 23:00:00
simulation_results: 9 rows
  week_start: 2024-02-01 00:00:00 to 2024-03-28 00:00:00
  week_end: 2024-02-08 00:00:00 to 2024-04-01 00:00:00
drift_metrics: 45 rows
  week_start: 2024-02-01 00:00:00 to 2024-03-28 00:00:00
  week_end: 2024-02-08 00:00:00 to 2024-04-01 00:00:00
retrain_events: 8 rows
  timestamp: 2024-02-08 00:00:00 to 2024-03-28 00:00:00
weekly_predictions: 28,800 rows
  timestamp: 2024-02-01 00:00:00 to 2024-03-31 23:00:00


## 2. Generate JSON Files

This section builds the eight frontend data contracts. All dates are serialized as ISO strings, all floats are rounded for display, and missing values are converted to JSON `null`.

In [3]:
FRONTEND_DATA_DIR.mkdir(parents=True, exist_ok=True)


def to_json_native(value: Any) -> Any:
    """Convert pandas/numpy objects and non-finite values to JSON-safe Python values."""
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, (datetime,)):
        return value.isoformat()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        value = float(value)
    if isinstance(value, float):
        if math.isnan(value) or math.isinf(value):
            return None
        return value
    if isinstance(value, np.ndarray):
        return [to_json_native(item) for item in value.tolist()]
    if isinstance(value, dict):
        return {str(key): to_json_native(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_json_native(item) for item in value]
    return value


def round_or_none(value: Any, digits: int = 2) -> float | None:
    """Round finite numeric values and return None for missing or non-finite values."""
    if pd.isna(value):
        return None
    number = float(value)
    if math.isnan(number) or math.isinf(number):
        return None
    return round(number, digits)


def iso_date(value: Any) -> str | None:
    """Return a YYYY-MM-DD date string from a datetime-like value."""
    if pd.isna(value):
        return None
    return pd.Timestamp(value).date().isoformat()


def iso_datetime(value: Any) -> str | None:
    """Return an ISO 8601 timestamp string without timezone conversion."""
    if pd.isna(value):
        return None
    return pd.Timestamp(value).isoformat()


def write_json(filename: str, payload: Dict[str, Any]) -> Path:
    """Write indented, strict JSON and return the output path."""
    output_path = FRONTEND_DATA_DIR / filename
    safe_payload = to_json_native(payload)
    output_path.write_text(
        json.dumps(safe_payload, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    print(f"Wrote {output_path}")
    return output_path

In [4]:
def smape(y_true: Iterable[float], y_pred: Iterable[float]) -> float:
    """Return symmetric mean absolute percentage error as a percentage."""
    y_true_series = pd.Series(y_true, dtype=float)
    y_pred_series = pd.Series(y_pred, dtype=float)
    denom = (y_true_series.abs() + y_pred_series.abs()).replace(0, 1e-8)
    return float((2 * (y_true_series - y_pred_series).abs() / denom).mean() * 100)


def mae(y_true: Iterable[float], y_pred: Iterable[float]) -> float:
    """Return mean absolute error."""
    return float((pd.Series(y_true, dtype=float) - pd.Series(y_pred, dtype=float)).abs().mean())


def rmse(y_true: Iterable[float], y_pred: Iterable[float]) -> float:
    """Return root mean squared error."""
    error = pd.Series(y_true, dtype=float) - pd.Series(y_pred, dtype=float)
    return float((error.pow(2).mean()) ** 0.5)


def wape(y_true: Iterable[float], y_pred: Iterable[float]) -> float:
    """Return weighted absolute percentage error as a percentage."""
    y_true_series = pd.Series(y_true, dtype=float)
    y_pred_series = pd.Series(y_pred, dtype=float)
    denominator = y_true_series.abs().sum()
    if denominator == 0:
        return float("nan")
    return float((y_true_series - y_pred_series).abs().sum() / denominator * 100)


def metric_block(frame: pd.DataFrame, prediction_column: str, actual_column: str = "actual_demand") -> Dict[str, float | None]:
    """Build a rounded metric block for one prediction column."""
    return {
        "smape": round_or_none(smape(frame[actual_column], frame[prediction_column])),
        "rmse": round_or_none(rmse(frame[actual_column], frame[prediction_column])),
        "mae": round_or_none(mae(frame[actual_column], frame[prediction_column])),
        "wape": round_or_none(wape(frame[actual_column], frame[prediction_column])),
    }

In [5]:
top_zone_demand = (
    weekly_predictions.groupby(["zone_id", "zone_name", "borough"], as_index=False)["actual_demand"]
    .sum()
    .sort_values("actual_demand", ascending=False)
    .reset_index(drop=True)
)
top_zone_ids = top_zone_demand["zone_id"].astype(int).tolist()

# Recompute demand coverage from the raw Jan-Mar files so the summary is derived from data, not constants.
raw_total_demand = 0
raw_top20_demand = 0
for trip_file in [
    RAW_DIR / "yellow_tripdata_2024-01.parquet",
    RAW_DIR / "yellow_tripdata_2024-02.parquet",
    RAW_DIR / "yellow_tripdata_2024-03.parquet",
]:
    raw_trip_part = pd.read_parquet(trip_file, columns=["tpep_pickup_datetime", "PULocationID"])
    raw_trip_part["tpep_pickup_datetime"] = pd.to_datetime(
        raw_trip_part["tpep_pickup_datetime"], errors="coerce"
    )
    raw_trip_part = raw_trip_part.dropna(subset=["tpep_pickup_datetime", "PULocationID"])
    raw_trip_part = raw_trip_part[
        (raw_trip_part["tpep_pickup_datetime"] >= pd.Timestamp("2024-01-01"))
        & (raw_trip_part["tpep_pickup_datetime"] < pd.Timestamp("2024-04-01"))
        & (raw_trip_part["PULocationID"] > 0)
    ]
    raw_trip_part["PULocationID"] = raw_trip_part["PULocationID"].astype(int)
    raw_total_demand += len(raw_trip_part)
    raw_top20_demand += int(raw_trip_part["PULocationID"].isin(top_zone_ids).sum())

demand_coverage_pct = raw_top20_demand / raw_total_demand * 100
print(f"Top 20 modeled zones: {len(top_zone_ids)}")
print(f"Demand coverage from raw Jan-Mar data: {demand_coverage_pct:.2f}%")

Top 20 modeled zones: 20
Demand coverage from raw Jan-Mar data: 63.05%


In [6]:
ZONE_COORDINATES = {
    161: {"latitude": 40.7587, "longitude": -73.9787},
    237: {"latitude": 40.7736, "longitude": -73.9566},
    132: {"latitude": 40.6413, "longitude": -73.7781},
    236: {"latitude": 40.7831, "longitude": -73.9507},
    162: {"latitude": 40.7558, "longitude": -73.9724},
    230: {"latitude": 40.7590, "longitude": -73.9851},
    186: {"latitude": 40.7484, "longitude": -73.9925},
    142: {"latitude": 40.7739, "longitude": -73.9820},
    138: {"latitude": 40.7769, "longitude": -73.8740},
    239: {"latitude": 40.7830, "longitude": -73.9788},
    163: {"latitude": 40.7643, "longitude": -73.9776},
    170: {"latitude": 40.7479, "longitude": -73.9757},
    234: {"latitude": 40.7359, "longitude": -73.9911},
    68: {"latitude": 40.7465, "longitude": -74.0014},
    48: {"latitude": 40.7638, "longitude": -73.9925},
    79: {"latitude": 40.7278, "longitude": -73.9853},
    249: {"latitude": 40.7344, "longitude": -74.0027},
    141: {"latitude": 40.7654, "longitude": -73.9614},
    164: {"latitude": 40.7485, "longitude": -73.9840},
    140: {"latitude": 40.7685, "longitude": -73.9586},
}

missing_coordinate_ids = [zone_id for zone_id in top_zone_ids if zone_id not in ZONE_COORDINATES]
if missing_coordinate_ids:
    raise ValueError(f"Missing coordinates for modeled zones: {missing_coordinate_ids}")

In [7]:
zone_metadata = top_zone_demand.merge(
    zone_lookup[["zone_id", "service_zone"]], on="zone_id", how="left"
)
zones_payload = {
    "zones": [],
    "boroughs": sorted(zone_lookup["borough"].dropna().unique().tolist()),
    "total_zones": int(len(zone_metadata)),
    "coordinate_note": "Approximate display centroids for TLC taxi zones.",
}
for rank, row in zone_metadata.iterrows():
    coordinates = ZONE_COORDINATES[int(row["zone_id"])]
    zones_payload["zones"].append(
        {
            "id": int(row["zone_id"]),
            "name": row["zone_name"],
            "borough": row["borough"],
            "service_zone": row["service_zone"],
            "total_demand": int(round(row["actual_demand"])),
            "rank": int(rank + 1),
            "latitude": round_or_none(coordinates["latitude"], 4),
            "longitude": round_or_none(coordinates["longitude"], 4),
        }
    )

write_json("zones.json", zones_payload)

Wrote ..\frontend\public\data\zones.json


WindowsPath('../frontend/public/data/zones.json')

In [8]:
prediction_metric_rows = []
baseline_columns = {
    "naive_lag1": "naive_pred",
    "seasonal_lag24": "seasonal_pred",
    "moving_avg": "moving_avg_pred",
    "lightgbm": "lightgbm_pred",
    "xgboost": "xgboost_pred",
}
for model_name, prediction_column in baseline_columns.items():
    prediction_metric_rows.append(
        {
            "model": model_name,
            "smape": smape(predictions_v1["actual_demand"], predictions_v1[prediction_column]),
            "rmse": rmse(predictions_v1["actual_demand"], predictions_v1[prediction_column]),
            "mae": mae(predictions_v1["actual_demand"], predictions_v1[prediction_column]),
            "wape": wape(predictions_v1["actual_demand"], predictions_v1[prediction_column]),
        }
    )
prediction_metrics = pd.DataFrame(prediction_metric_rows)
best_baseline = prediction_metrics[prediction_metrics["model"].isin(["naive_lag1", "seasonal_lag24", "moving_avg"])].sort_values("smape").iloc[0]
lightgbm_metrics = prediction_metrics[prediction_metrics["model"] == "lightgbm"].iloc[0]

simulation_results = simulation_results.sort_values("week_start").reset_index(drop=True)
simulation_results["adaptive_improvement_pct"] = (
    (simulation_results["static_smape"] - simulation_results["adaptive_smape"])
    / simulation_results["static_smape"]
    * 100
)
static_avg_smape = float(simulation_results["static_smape"].mean())
adaptive_avg_smape = float(simulation_results["adaptive_smape"].mean())
simulation_improvement_pct = (static_avg_smape - adaptive_avg_smape) / static_avg_smape * 100
t_statistic, p_value = stats.ttest_rel(
    simulation_results["static_smape"],
    simulation_results["adaptive_smape"],
    alternative="greater",
)
best_week = simulation_results.sort_values("adaptive_improvement_pct", ascending=False).iloc[0]
worst_week = simulation_results.sort_values("adaptive_improvement_pct").iloc[0]
ordered_features = drift_metrics.sort_values(["week_start"]).groupby("feature", sort=False).size().index.tolist()
current_version = weekly_predictions.sort_values("timestamp")["model_version"].iloc[-1]
latest_week = drift_metrics["week_start"].max()
latest_max_psi = float(drift_metrics[drift_metrics["week_start"] == latest_week]["psi"].max())
drift_status = "alert" if latest_max_psi >= 0.2 else "warning" if latest_max_psi >= 0.1 else "healthy"

summary_payload = {
    "model_performance": {
        "test_smape": round_or_none(lightgbm_metrics["smape"]),
        "test_rmse": round_or_none(lightgbm_metrics["rmse"]),
        "test_mae": round_or_none(lightgbm_metrics["mae"]),
        "test_wape": round_or_none(lightgbm_metrics["wape"]),
        "improvement_over_baseline": round_or_none((best_baseline["smape"] - lightgbm_metrics["smape"]) / best_baseline["smape"] * 100),
        "best_baseline": best_baseline["model"],
        "best_baseline_smape": round_or_none(best_baseline["smape"]),
    },
    "simulation_results": {
        "static_avg_smape": round_or_none(static_avg_smape),
        "adaptive_avg_smape": round_or_none(adaptive_avg_smape),
        "improvement_pct": round_or_none(simulation_improvement_pct),
        "p_value": round_or_none(p_value, 4),
        "is_significant": bool(p_value < 0.05),
        "total_weeks": int(len(simulation_results)),
        "total_retrains": int(len(retrain_events)),
        "best_week": {
            "date": iso_date(best_week["week_start"]),
            "improvement": round_or_none(best_week["adaptive_improvement_pct"]),
        },
        "worst_week": {
            "date": iso_date(worst_week["week_start"]),
            "improvement": round_or_none(worst_week["adaptive_improvement_pct"]),
        },
    },
    "data_info": {
        "training_period": "2024-01-01 to 2024-01-30",
        "simulation_period": "2024-02-01 to 2024-03-31",
        "total_predictions": int(len(weekly_predictions)),
        "zones_count": int(len(top_zone_ids)),
        "demand_coverage_pct": round_or_none(demand_coverage_pct),
    },
    "model_info": {
        "model_type": "LightGBM",
        "top_features": ordered_features[:5],
        "trained_at": "2024-01-30",
        "current_version": current_version,
        "drift_status": drift_status,
    },
}

write_json("summary.json", summary_payload)

Wrote ..\frontend\public\data\summary.json


WindowsPath('../frontend/public/data/summary.json')

In [9]:
last_timestamp = weekly_predictions["timestamp"].max()
prediction_start = last_timestamp - pd.Timedelta(days=7)
recent_predictions = weekly_predictions[weekly_predictions["timestamp"] >= prediction_start].copy()
recent_predictions = recent_predictions.sort_values(["zone_id", "timestamp"])

predictions_payload = {"zones": {}}
for zone_id, zone_frame in recent_predictions.groupby("zone_id"):
    zone_frame = zone_frame.sort_values("timestamp")
    predictions_payload["zones"][str(int(zone_id))] = {
        "zone_name": zone_frame["zone_name"].iloc[0],
        "borough": zone_frame["borough"].iloc[0],
        "predictions": [
            {
                "timestamp": iso_datetime(row["timestamp"]),
                "actual": round_or_none(row["actual_demand"]),
                "static_pred": round_or_none(row["static_prediction"]),
                "adaptive_pred": round_or_none(row["adaptive_prediction"]),
            }
            for _, row in zone_frame.iterrows()
        ],
    }

write_json("predictions.json", predictions_payload)
print(f"predictions.json covers {prediction_start} to {last_timestamp}")

Wrote ..\frontend\public\data\predictions.json
predictions.json covers 2024-03-24 23:00:00 to 2024-03-31 23:00:00


In [10]:
feature_order = ordered_features[:5]
drift_timeseries = []
for week_start, week_frame in drift_metrics.sort_values(["week_start", "feature"]).groupby("week_start"):
    week_by_feature = week_frame.set_index("feature")
    psi_scores = {
        feature: round_or_none(week_by_feature.loc[feature, "psi"], 4)
        for feature in feature_order
        if feature in week_by_feature.index
    }
    ks_p_values = {
        feature: round_or_none(week_by_feature.loc[feature, "ks_p_value"], 4)
        for feature in feature_order
        if feature in week_by_feature.index
    }
    drift_timeseries.append(
        {
            "week_start": iso_date(week_start),
            "psi_scores": psi_scores,
            "ks_p_values": ks_p_values,
            "drift_detected": bool(week_frame["drift_detected"].any() or (week_frame["psi"] > 0.2).any()),
            "max_psi": round_or_none(week_frame["psi"].max(), 4),
        }
    )

drift_payload = {
    "features": feature_order,
    "thresholds": {"warning": 0.1, "alert": 0.2},
    "timeseries": drift_timeseries,
}
write_json("drift_metrics.json", drift_payload)

Wrote ..\frontend\public\data\drift_metrics.json


WindowsPath('../frontend/public/data/drift_metrics.json')

In [11]:
def retrain_improvement_after(timestamp: pd.Timestamp) -> float | None:
    """Compare the simulation week before a retrain with the first available week after it."""
    before_rows = simulation_results[simulation_results["week_end"] <= timestamp].tail(1)
    after_rows = simulation_results[simulation_results["week_start"] >= timestamp].head(1)
    if before_rows.empty or after_rows.empty:
        return None
    before_smape = float(before_rows["adaptive_smape"].mean())
    after_smape = float(after_rows["adaptive_smape"].mean())
    if before_smape == 0:
        return None
    return (before_smape - after_smape) / before_smape * 100


retrain_events = retrain_events.sort_values("timestamp").reset_index(drop=True)
retrain_payload_events = []
for index, row in retrain_events.iterrows():
    retrain_payload_events.append(
        {
            "id": int(index + 1),
            "timestamp": iso_datetime(row["timestamp"]),
            "trigger_reason": row["trigger_reason"],
            "model_version": row["model_version"],
            "training_size": int(row["training_data_size"]),
            "training_duration_seconds": round_or_none(row["training_duration_seconds"]),
            "improvement_after": round_or_none(retrain_improvement_after(row["timestamp"])),
        }
    )
if len(retrain_events) > 1:
    avg_days_between = retrain_events["timestamp"].diff().dt.total_seconds().dropna().mean() / 86400
else:
    avg_days_between = None

retrain_payload = {
    "events": retrain_payload_events,
    "total_events": int(len(retrain_payload_events)),
    "avg_days_between_retrains": round_or_none(avg_days_between),
}
write_json("retrain_events.json", retrain_payload)

Wrote ..\frontend\public\data\retrain_events.json


WindowsPath('../frontend/public/data/retrain_events.json')

In [12]:
week_entries = []
for _, row in simulation_results.iterrows():
    inclusive_week_end = pd.Timestamp(row["week_end"]) - pd.Timedelta(days=1)
    week_entries.append(
        {
            "week_start": iso_date(row["week_start"]),
            "week_end": iso_date(inclusive_week_end),
            "static": {
                "smape": round_or_none(row["static_smape"]),
                "rmse": round_or_none(row["static_rmse"]),
                "mae": round_or_none(row["static_mae"]),
                "wape": round_or_none(row["static_wape"]),
            },
            "adaptive": {
                "smape": round_or_none(row["adaptive_smape"]),
                "rmse": round_or_none(row["adaptive_rmse"]),
                "mae": round_or_none(row["adaptive_mae"]),
                "wape": round_or_none(row["adaptive_wape"]),
            },
            "improvement_smape": round_or_none(row["adaptive_improvement_pct"]),
            "model_version_used": row["model_version"],
            "retrain_occurred": bool(row["retrain_triggered"]),
        }
    )

weekly_payload = {
    "weeks": week_entries,
    "cumulative": {
        "static_smape_progression": [
            round_or_none(value) for value in simulation_results["static_smape"].expanding().mean()
        ],
        "adaptive_smape_progression": [
            round_or_none(value) for value in simulation_results["adaptive_smape"].expanding().mean()
        ],
        "dates": [iso_date(value) for value in simulation_results["week_start"]],
    },
}
write_json("weekly_comparison.json", weekly_payload)

Wrote ..\frontend\public\data\weekly_comparison.json


WindowsPath('../frontend/public/data/weekly_comparison.json')

In [13]:
zone_rows = []
for (zone_id, zone_name, borough), zone_frame in weekly_predictions.groupby(["zone_id", "zone_name", "borough"]):
    static_zone_smape = smape(zone_frame["actual_demand"], zone_frame["static_prediction"])
    adaptive_zone_smape = smape(zone_frame["actual_demand"], zone_frame["adaptive_prediction"])
    zone_rows.append(
        {
            "zone_id": int(zone_id),
            "zone_name": zone_name,
            "borough": borough,
            "metrics": {
                "static_smape": round_or_none(static_zone_smape),
                "adaptive_smape": round_or_none(adaptive_zone_smape),
                "static_rmse": round_or_none(rmse(zone_frame["actual_demand"], zone_frame["static_prediction"])),
                "adaptive_rmse": round_or_none(rmse(zone_frame["actual_demand"], zone_frame["adaptive_prediction"])),
                "improvement_pct": round_or_none((static_zone_smape - adaptive_zone_smape) / static_zone_smape * 100 if static_zone_smape else None),
                "total_predictions": int(len(zone_frame)),
                "avg_actual_demand": round_or_none(zone_frame["actual_demand"].mean()),
            },
        }
    )

borough_payload = {}
for borough, borough_frame in weekly_predictions.groupby("borough"):
    static_borough_smape = smape(borough_frame["actual_demand"], borough_frame["static_prediction"])
    adaptive_borough_smape = smape(borough_frame["actual_demand"], borough_frame["adaptive_prediction"])
    borough_payload[borough] = {
        "avg_static_smape": round_or_none(static_borough_smape),
        "avg_adaptive_smape": round_or_none(adaptive_borough_smape),
        "improvement": round_or_none((static_borough_smape - adaptive_borough_smape) / static_borough_smape * 100 if static_borough_smape else None),
        "zone_count": int(borough_frame["zone_id"].nunique()),
    }

zone_performance_payload = {
    "zones": sorted(zone_rows, key=lambda item: item["metrics"]["adaptive_smape"]),
    "by_borough": borough_payload,
}
write_json("zone_performance.json", zone_performance_payload)

Wrote ..\frontend\public\data\zone_performance.json


WindowsPath('../frontend/public/data/zone_performance.json')

In [14]:
time_frame = weekly_predictions.copy()
time_frame["hour"] = time_frame["timestamp"].dt.hour
time_frame["day_of_week"] = time_frame["timestamp"].dt.dayofweek
time_frame["is_weekend"] = time_frame["day_of_week"].isin([5, 6])
time_frame["is_rush_hour"] = (
    time_frame["day_of_week"].between(0, 4)
    & (time_frame["hour"].between(7, 9) | time_frame["hour"].between(17, 19))
)

def segment_metrics(frame: pd.DataFrame, definition: str) -> Dict[str, Any]:
    """Return frontend-ready metrics for a time segment."""
    static_segment_smape = smape(frame["actual_demand"], frame["static_prediction"])
    adaptive_segment_smape = smape(frame["actual_demand"], frame["adaptive_prediction"])
    return {
        "static_smape": round_or_none(static_segment_smape),
        "adaptive_smape": round_or_none(adaptive_segment_smape),
        "improvement": round_or_none((static_segment_smape - adaptive_segment_smape) / static_segment_smape * 100 if static_segment_smape else None),
        "definition": definition,
    }

by_hour = []
for hour, hour_frame in time_frame.groupby("hour"):
    by_hour.append(
        {
            "hour": int(hour),
            "static_smape": round_or_none(smape(hour_frame["actual_demand"], hour_frame["static_prediction"])),
            "adaptive_smape": round_or_none(smape(hour_frame["actual_demand"], hour_frame["adaptive_prediction"])),
            "avg_demand": round_or_none(hour_frame["actual_demand"].mean()),
        }
    )

time_period_payload = {
    "rush_hour": segment_metrics(time_frame[time_frame["is_rush_hour"]], "7-9 AM and 5-7 PM weekdays"),
    "off_peak": segment_metrics(time_frame[~time_frame["is_rush_hour"]], "All non-rush-hour periods"),
    "weekend": segment_metrics(time_frame[time_frame["is_weekend"]], "Saturday and Sunday"),
    "weekday": segment_metrics(time_frame[~time_frame["is_weekend"]], "Monday through Friday"),
    "by_hour": sorted(by_hour, key=lambda item: item["hour"]),
}
write_json("time_period_analysis.json", time_period_payload)

Wrote ..\frontend\public\data\time_period_analysis.json


WindowsPath('../frontend/public/data/time_period_analysis.json')

## 3. NYC Zone Geographic Coordinates

`zones.json` includes approximate latitude/longitude display centroids for the actual top-20 modeled taxi zones. These coordinates are sufficient for frontend visualization markers; exact polygons would require the TLC taxi-zone shapefile.

In [15]:
zones_with_coordinates = json.loads((FRONTEND_DATA_DIR / "zones.json").read_text(encoding="utf-8"))
coordinate_check = [
    zone for zone in zones_with_coordinates["zones"] if zone.get("latitude") is None or zone.get("longitude") is None
]
if coordinate_check:
    raise ValueError(f"Zones missing coordinates: {coordinate_check}")
print("All top-20 modeled zones have latitude/longitude in zones.json.")

All top-20 modeled zones have latitude/longitude in zones.json.


## 4. File Size Verification

The React app should be able to load these files quickly as static assets. The total export size target is under 5 MB.

In [16]:
json_paths = [FRONTEND_DATA_DIR / filename for filename in JSON_FILES]
total_size_bytes = 0
for path in json_paths:
    size_bytes = path.stat().st_size
    total_size_bytes += size_bytes
    print(f"{path}: {size_bytes / 1024:.2f} KB")

print(f"Total frontend/public/data size: {total_size_bytes / 1024:.2f} KB")
assert total_size_bytes < 5 * 1024 * 1024, "Frontend data export exceeds 5 MB."

..\frontend\public\data\zones.json: 4.98 KB
..\frontend\public\data\summary.json: 1.12 KB
..\frontend\public\data\predictions.json: 548.37 KB
..\frontend\public\data\drift_metrics.json: 3.93 KB
..\frontend\public\data\retrain_events.json: 2.53 KB
..\frontend\public\data\weekly_comparison.json: 4.47 KB
..\frontend\public\data\zone_performance.json: 7.63 KB
..\frontend\public\data\time_period_analysis.json: 3.50 KB
Total frontend/public/data size: 576.54 KB


## 5. Validation

Each JSON file is loaded back from disk to confirm parseability and required top-level keys. A short preview is printed for quick structure inspection.

In [17]:
REQUIRED_KEYS = {
    "zones.json": ["zones", "boroughs", "total_zones"],
    "summary.json": ["model_performance", "simulation_results", "data_info", "model_info"],
    "predictions.json": ["zones"],
    "drift_metrics.json": ["features", "thresholds", "timeseries"],
    "retrain_events.json": ["events", "total_events", "avg_days_between_retrains"],
    "weekly_comparison.json": ["weeks", "cumulative"],
    "zone_performance.json": ["zones", "by_borough"],
    "time_period_analysis.json": ["rush_hour", "off_peak", "weekend", "weekday", "by_hour"],
}

for filename, required_keys in REQUIRED_KEYS.items():
    path = FRONTEND_DATA_DIR / filename
    with path.open("r", encoding="utf-8") as file:
        payload = json.load(file)
    missing_keys = [key for key in required_keys if key not in payload]
    if missing_keys:
        raise ValueError(f"{filename} missing required keys: {missing_keys}")
    preview = json.dumps(payload, indent=2)[:200]
    print(f"{filename}: valid JSON")
    print(preview.replace("\n", " ") + "...")

zones.json: valid JSON
{   "zones": [     {       "id": 161,       "name": "Midtown Center",       "borough": "Manhattan",       "service_zone": "Yellow Zone",       "total_demand": 310356,       "rank": 1,       "latitude"...
summary.json: valid JSON
{   "model_performance": {     "test_smape": 22.78,     "test_rmse": 37.17,     "test_mae": 24.57,     "test_wape": 16.33,     "improvement_over_baseline": 46.1,     "best_baseline": "seasonal_lag24",...
predictions.json: valid JSON
{   "zones": {     "48": {       "zone_name": "Clinton East",       "borough": "Manhattan",       "predictions": [         {           "timestamp": "2024-03-24T23:00:00",           "actual": 78.0,    ...


drift_metrics.json: valid JSON
{   "features": [     "lag_168h",     "hour",     "lag_24h",     "lag_1h",     "zone_id"   ],   "thresholds": {     "warning": 0.1,     "alert": 0.2   },   "timeseries": [     {       "week_start": "2...
retrain_events.json: valid JSON
{   "events": [     {       "id": 1,       "timestamp": "2024-02-08T00:00:00",       "trigger_reason": "adaptive sMAPE >= 20% above reference baseline for 3 consecutive days",       "model_version": "...
weekly_comparison.json: valid JSON
{   "weeks": [     {       "week_start": "2024-02-01",       "week_end": "2024-02-07",       "static": {         "smape": 25.26,         "rmse": 31.89,         "mae": 20.95,         "wape": 15.67     ...
zone_performance.json: valid JSON
{   "zones": [     {       "zone_id": 48,       "zone_name": "Clinton East",       "borough": "Manhattan",       "metrics": {         "static_smape": 20.32,         "adaptive_smape": 18.53,         "s...
time_period_analysis.json: valid JSON
{   "rush_hour

## 6. Generate API Contract Document

The frontend contract README describes each static JSON file, its purpose, its TypeScript-like schema, and when it should be regenerated.

In [18]:
contract_markdown = """# Frontend Static Data Contracts

These files are generated by `notebooks/04_export_for_frontend.ipynb` from the processed modeling outputs. They are static JSON assets intended to be fetched by the React app from `/data/<filename>.json`.

## Update Frequency

Regenerate these files whenever `data/processed/*.csv` changes, when a new model is trained, or when the frontend needs a refreshed demo snapshot.

## `zones.json`

Purpose: top modeled taxi zones, ranks, demand totals, service-zone metadata, and approximate map coordinates.

```ts
interface ZonesJson {
  zones: Array<{
    id: number;
    name: string;
    borough: string;
    service_zone: string;
    total_demand: number;
    rank: number;
    latitude: number;
    longitude: number;
  }>;
  boroughs: string[];
  total_zones: number;
  coordinate_note: string;
}
```

Example: `zones[0]` contains the highest-demand modeled zone.

## `summary.json`

Purpose: dashboard headline metrics for model test performance, production simulation performance, data coverage, and current model status.

```ts
interface SummaryJson {
  model_performance: {
    test_smape: number;
    test_rmse: number;
    test_mae: number;
    test_wape: number;
    improvement_over_baseline: number;
    best_baseline: string;
    best_baseline_smape: number;
  };
  simulation_results: {
    static_avg_smape: number;
    adaptive_avg_smape: number;
    improvement_pct: number;
    p_value: number;
    is_significant: boolean;
    total_weeks: number;
    total_retrains: number;
    best_week: { date: string; improvement: number };
    worst_week: { date: string; improvement: number };
  };
  data_info: {
    training_period: string;
    simulation_period: string;
    total_predictions: number;
    zones_count: number;
    demand_coverage_pct: number;
  };
  model_info: {
    model_type: string;
    top_features: string[];
    trained_at: string;
    current_version: string;
    drift_status: "healthy" | "warning" | "alert";
  };
}
```

## `predictions.json`

Purpose: last seven days of hourly static and adaptive predictions by zone.

```ts
interface PredictionsJson {
  zones: Record<string, {
    zone_name: string;
    borough: string;
    predictions: Array<{
      timestamp: string;
      actual: number | null;
      static_pred: number | null;
      adaptive_pred: number | null;
    }>;
  }>;
}
```

## `drift_metrics.json`

Purpose: weekly PSI and KS-test drift signals for important model features.

```ts
interface DriftMetricsJson {
  features: string[];
  thresholds: { warning: number; alert: number };
  timeseries: Array<{
    week_start: string;
    psi_scores: Record<string, number | null>;
    ks_p_values: Record<string, number | null>;
    drift_detected: boolean;
    max_psi: number | null;
  }>;
}
```

## `retrain_events.json`

Purpose: retraining timeline and model version changes.

```ts
interface RetrainEventsJson {
  events: Array<{
    id: number;
    timestamp: string;
    trigger_reason: string;
    model_version: string;
    training_size: number;
    training_duration_seconds: number | null;
    improvement_after: number | null;
  }>;
  total_events: number;
  avg_days_between_retrains: number | null;
}
```

## `weekly_comparison.json`

Purpose: week-by-week static versus adaptive model comparison and cumulative sMAPE progression.

```ts
interface WeeklyComparisonJson {
  weeks: Array<{
    week_start: string;
    week_end: string;
    static: { smape: number; rmse: number; mae: number; wape: number };
    adaptive: { smape: number; rmse: number; mae: number; wape: number };
    improvement_smape: number;
    model_version_used: string;
    retrain_occurred: boolean;
  }>;
  cumulative: {
    static_smape_progression: number[];
    adaptive_smape_progression: number[];
    dates: string[];
  };
}
```

## `zone_performance.json`

Purpose: per-zone and per-borough forecast error comparisons.

```ts
interface ZonePerformanceJson {
  zones: Array<{
    zone_id: number;
    zone_name: string;
    borough: string;
    metrics: {
      static_smape: number;
      adaptive_smape: number;
      static_rmse: number;
      adaptive_rmse: number;
      improvement_pct: number;
      total_predictions: number;
      avg_actual_demand: number;
    };
  }>;
  by_borough: Record<string, {
    avg_static_smape: number;
    avg_adaptive_smape: number;
    improvement: number;
    zone_count: number;
  }>;
}
```

## `time_period_analysis.json`

Purpose: performance by rush hour, off peak, weekend, weekday, and hour of day.

```ts
interface TimePeriodAnalysisJson {
  rush_hour: SegmentMetrics;
  off_peak: SegmentMetrics;
  weekend: SegmentMetrics;
  weekday: SegmentMetrics;
  by_hour: Array<{
    hour: number;
    static_smape: number;
    adaptive_smape: number;
    avg_demand: number;
  }>;
}

interface SegmentMetrics {
  static_smape: number;
  adaptive_smape: number;
  improvement: number;
  definition: string;
}
```
"""

readme_path = FRONTEND_DATA_DIR / "README.md"
readme_path.write_text(dedent(contract_markdown), encoding="utf-8")
print(f"Wrote {readme_path}")

Wrote ..\frontend\public\data\README.md


## 7. Summary

The final cell prints the generated file tree and confirms that the static data directory is ready for React integration.

In [19]:
all_data_paths = sorted(FRONTEND_DATA_DIR.iterdir())
json_data_paths = [path for path in all_data_paths if path.suffix == ".json"]
total_bytes_written = sum(path.stat().st_size for path in all_data_paths if path.is_file())

print("✅ Generated 8 JSON files for frontend consumption")
for path in all_data_paths:
    if path.is_file():
        print(f"- {path.name}: {path.stat().st_size / 1024:.2f} KB")
print(f"Total bytes written: {total_bytes_written:,}")
print("\nfrontend/public/data/")
for path in all_data_paths:
    print(f"  {path.name}")

assert len(json_data_paths) == 8, f"Expected 8 JSON files, found {len(json_data_paths)}"
print("✅ Frontend data export complete. Ready for React integration.")

✅ Generated 8 JSON files for frontend consumption
- drift_metrics.json: 3.93 KB
- predictions.json: 548.37 KB
- README.md: 5.03 KB
- retrain_events.json: 2.53 KB
- summary.json: 1.12 KB
- time_period_analysis.json: 3.50 KB
- weekly_comparison.json: 4.47 KB
- zone_performance.json: 7.63 KB
- zones.json: 4.98 KB
Total bytes written: 595,520

frontend/public/data/
  drift_metrics.json
  predictions.json
  README.md
  retrain_events.json
  summary.json
  time_period_analysis.json
  weekly_comparison.json
  zone_performance.json
  zones.json
✅ Frontend data export complete. Ready for React integration.
